In [ ]:
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sys.path.append(os.path.abspath(os.path.join("..", "")))
from Data.clean_data import get_ai_developer_dataset
df = get_ai_developer_dataset()
df

In [ ]:
y = df['Task_Duration_Hours']
feature_cols = [
    'Hours_Coding', 
    'Lines_of_Code', 
    'Bugs_Found', 
    'AI_Usage_Hours', 
    'Sleep_Hours', 
    'Cognitive_Load', 
    'Coffee_Intake', 
    'Stress_Level', 
    'Commits'
]
X = df[feature_cols]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

r2_score(y_test, y_pred_lr)

In [ ]:
print(f"Mean Absolute Error (MAE): {mae_lr:.2f} hours")
print(f"Root Mean Squared Error (RMSE): {rmse_lr:.2f} hours")
print(f"R-squared (R2) Score: {r2_lr:.4f}")

## Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train) 

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_score(y_test, y_pred_rf)


In [ ]:
r2_rf = r2_score(y_test, y_pred_rf)
print("--- Random Forest Results ---")
print(f"Mean Absolute Error (MAE): {mae_rf:.2f} hours")
print(f"Root Mean Squared Error (RMSE): {rmse_rf:.2f} hours")
print(f"R-squared (R2) Score: {r2_rf:.4f}")

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=np.array(feature_cols)[indices], palette='viridis')
plt.title('Feature Importance for Predicting Task Duration (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()


In [ ]:
coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_
}).sort_values(by='Coefficient', ascending=False)
print(coefficients.to_string(index=False))


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.6, color='dodgerblue', edgecolors='k')

min_val = min(y_test.min(), y_pred_lr.min())
max_val = max(y_test.max(), y_pred_lr.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction (y = x)')

plt.title('Model Evaluation: Actual vs. Predicted Task Duration')
plt.xlabel('Actual Task Duration (Hours)')
plt.ylabel('Predicted Task Duration (Hours)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
residuals = y_test - y_pred_lr

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(x=y_pred_lr, y=residuals, ax=axes[0], alpha=0.6, color='purple')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_title('Residuals vs. Predicted Values')
axes[0].set_xlabel('Predicted Task Duration')
axes[0].set_ylabel('Residuals (Errors)')

sns.histplot(residuals, kde=True, ax=axes[1], color='teal', bins=20)
axes[1].set_title('Distribution of Residuals')
axes[1].set_xlabel('Residual Error (Hours)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
df_eng = df.copy()

df_eng['Total_Active_Hours'] = df_eng['Hours_Coding'] + df_eng['AI_Usage_Hours']
df_eng['AI_to_Manual_Ratio'] = df_eng['AI_Usage_Hours'] / (df_eng['Hours_Coding'] + 0.1) # Add 0.1 to avoid division by zero
df_eng['Lines_per_Hour'] = df_eng['Lines_of_Code'] / (df_eng['Hours_Coding'] + 0.1)
df_eng['Bugs_per_Line'] = df_eng['Bugs_Found'] / df_eng['Lines_of_Code']

expanded_features = feature_cols + [
    'Bugs_Fixed', 
    'Errors', 
    'Total_Active_Hours', 
    'AI_to_Manual_Ratio', 
    'Lines_per_Hour', 
    'Bugs_per_Line'
]

X_eng = df_eng[expanded_features]
y_eng = df_eng['Task_Duration_Hours']

X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(X_eng, y_eng, test_size=0.2, random_state=42)

scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

print(f"New Feature Set Shape: {X_eng.shape}")


## Gradient Boosting Regressor 

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model = GradientBoostingRegressor(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=4, 
    subsample=0.8, 
    random_state=42
)

gbr_model.fit(X_train_eng, y_train_eng)

y_pred_gbr = gbr_model.predict(X_test_eng)
mae_gbr = mean_absolute_error(y_test_eng, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test_eng, y_pred_gbr))

r2_score(y_test_eng, y_pred_gbr)

In [ ]:
r2_gbr = r2_score(y_test_eng, y_pred_gbr)

print("--- Gradient Boosting Regressor Results ---")
print(f"Mean Absolute Error (MAE): {mae_gbr:.2f} hours")
print(f"Root Mean Squared Error (RMSE): {rmse_gbr:.2f} hours")
print(f"R-squared (R2) Score: {r2_gbr:.4f}")